# Harness Engineering Demo: Better Harness Beats Bigger Model

This notebook is built for a management-facing demo.

**Thesis:** a medium model with a good harness can beat a great model with a bad harness, because production quality depends on context, tools, validation, memory, safety gates, observability, and repair loops, not only raw model capability.

We will show the spectrum:

1. **Great model + bad harness**: bare prompt, no shared memory, no runbook enforcement.
2. **Medium model + hand-built harness**: explicit agents, shared memory, sensors, and scorecard.
3. **Medium model + SDK harness**: Strands-style abstraction for tools, hooks, memory, and multi-agent orchestration.
4. **Provider plug-and-play harness**: provider-specific harness lane, represented by DeepSeek adapter boundary.

The reliable part of the demo is deterministic. The live model section calls Ollama Cloud so management can see how this connects to real model backends.

## Demo Architecture

```text
Colab notebook
      ↓
Harness demo repo + Python SDKs
      ↓
Scenario: multi-agent incident response
      ↓
Scorecard: evidence, runbook, safety, memory, completeness
      ↓
Optional live calls to Ollama Cloud
```

Colab is the runtime. Ollama Cloud is the model backend.

## 1. Clone The Repo

Replace `REPO_URL` with your GitHub URL after pushing the project.

If you already uploaded this notebook into the cloned repo, skip this cell and `%cd` into the repo folder.

In [ ]:
# Replace this with your pushed GitHub repository URL.
REPO_URL = "https://github.com/YOUR_ORG/ollama-harness-engineering-demo.git"
REPO_DIR = "ollama-harness-engineering-demo"

from pathlib import Path
import os

# If we are not already inside the repo, clone it or move into an existing clone.
if not Path("pyproject.toml").exists():
    if Path(REPO_DIR).exists():
        os.chdir(REPO_DIR)
    else:
        if "YOUR_ORG" in REPO_URL:
            raise ValueError(
                "Replace REPO_URL with your GitHub repo URL, then rerun this cell. "
                "Example: https://github.com/my-org/ollama-harness-engineering-demo.git"
            )
        !git clone $REPO_URL
        os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())
print("Project files:")
!ls -la

assert Path("requirements.txt").exists(), "requirements.txt not found. You are not inside the repo."
assert Path("pyproject.toml").exists(), "pyproject.toml not found. You are not inside the repo."

## 2. Install The Demo Dependencies

This happens inside Colab, so it does not depend on your office Mac allowing Python packages.

The important libraries are:

- `strands-agents`: SDK-level harness abstraction.
- `ollama`: direct calls to Ollama Cloud.
- `openai`: useful for OpenAI-compatible endpoints.
- `typer` and `rich`: CLI and readable scorecards.
- `pytest`: quick health check.

In [ ]:
from pathlib import Path

assert Path("requirements.txt").exists(), "Run the clone/%cd setup cell first. requirements.txt is missing here."
assert Path("pyproject.toml").exists(), "Run the clone/%cd setup cell first. pyproject.toml is missing here."

!pip install -r requirements.txt
!pip install -e .

## 3. Import The Repo Harness

Colab is only the presentation surface. The actual use case and harness workflow are imported from the repo.

No static lane outputs are used in this notebook. The management demo path below calls Ollama Cloud live and scores the actual model-generated artifacts.

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

repo_src = str(Path.cwd() / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

assert Path("src/harness_demo").exists(), "Not in repo root or src/harness_demo is missing."

from rich.console import Console
from rich.table import Table

from harness_demo.live import run_live_hand_built_lane, run_live_raw_lane
from harness_demo.reporting import print_comparison, print_result
from harness_demo.scenarios import load_incident_scenario

console = Console(width=110)
scenario = load_incident_scenario("incident-response")

print("Loaded scenario:", scenario.id)
print("Scenario name:", scenario.name)

## 4. The Use Case: Incident Response, Not Prompt Comparison

The scenario is a production checkout incident after a promotion launch.

A management audience should see the difference between:

- a model answering from a ticket alone
- an AI workflow that uses controlled evidence, approved runbooks, shared memory, reviewer checks, and repair

This is the point of harness engineering: the system around the model turns an answer into a governed workflow result.

In [ ]:
print("INCIDENT TICKET")
pprint(scenario.incident)

print("\nQUALITY CONTRACT USED BY THE SCORER")
pprint(scenario.expected)

## 5. Controlled Context Available To The Harness

These are not pasted wholesale into the raw model call.

They are available to the **harnessed workflow** through separate controlled steps:

- log investigator sees logs
- runbook agent sees runbook
- memory agent sees prior incident memory
- planner sees the shared memory created by earlier agents
- reviewer sees the final plan and checks policy/completeness

That separation is the harness. It is what changes between the weak and strong setup.

In [ ]:
print("LOG TOOL DATA")
print(scenario.logs)

print("RUNBOOK TOOL DATA")
print(scenario.runbook)

print("PRIOR MEMORY TOOL DATA")
print(scenario.prior_memory)

## 6. What Makes The Weak Harness Weak?

In this use case, the weak harness is weak because it has no operating controls:

| Concern | Weak harness behavior |
| --- | --- |
| Evidence | Only sees the incident ticket, not logs as a controlled tool. |
| Runbook | Does not receive or enforce approved remediation policy. |
| Memory | Does not use prior incident memory. |
| Safety | Has no reviewer gate to block unsafe operations. |
| Output contract | Can answer in any shape; no required fields. |
| Repair | No loop that feeds validation failures back into the workflow. |

So even if the model is strong, it is being used like a chat assistant, not a production workflow.

## 7. What Makes The Good Harness Good?

The good harness changes the execution environment around the model:

| Concern | Good harness behavior in this demo |
| --- | --- |
| Evidence | Separate log-investigator call receives logs and must extract evidence. |
| Runbook | Separate runbook call receives only approved runbook and extracts constraints. |
| Memory | Prior incident memory is queried as a separate controlled step. |
| Shared state | Findings are accumulated into `SharedMemory`. |
| Planning | Planner receives structured memory and required output fields. |
| Safety | Reviewer checks forbidden actions and missing fields. |
| Repair | If reviewer finds issues, a repair call revises the plan. |
| Measurement | Scorecard checks actual generated artifacts against the quality contract. |

The medium model does not win because of a magic prompt. It wins because the harness changes what the model is allowed to see, do, forget, and output.

## 8. Configure Ollama Cloud

Ollama Cloud is the live backend. The repo harness is the application workflow.

Do not hardcode the key in the notebook.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OLLAMA_API_KEY"):
    os.environ["OLLAMA_API_KEY"] = getpass("Enter OLLAMA_API_KEY: ")

print("OLLAMA_API_KEY configured:", bool(os.environ.get("OLLAMA_API_KEY")))

## 9. Choose Models

The intended comparison is asymmetric on purpose:

- stronger model with weak harness
- medium model with strong harness

Change these model names based on your Ollama Cloud subscription.

In [ ]:
RAW_STRONG_MODEL = "gpt-oss:120b"
HARNESS_MODEL = "gpt-oss:20b"

print("Strong model, weak harness:", RAW_STRONG_MODEL)
print("Medium model, good harness:", HARNESS_MODEL)

# Live Run 1: Strong Model + Weak Harness

This cell makes one live Ollama Cloud call.

The model receives the incident ticket only. It does not get the log tool, runbook tool, prior memory, reviewer, output contract, or repair loop.

The score is computed from the actual model output.

In [ ]:
live_raw = run_live_raw_lane(scenario, model_name=RAW_STRONG_MODEL)
print_result(console, live_raw)

print("\nACTUAL MODEL OUTPUT")
print(live_raw.final_answer)

print("\nWHAT THE SCORER COULD EXTRACT FROM THAT OUTPUT")
pprint(live_raw.memory)

# Live Run 2: Medium Model + Good Harness

This cell calls Ollama Cloud multiple times through the repo harness workflow.

Each call has a specific role and a controlled context boundary:

1. log investigator: incident + logs
2. runbook agent: incident + runbook
3. memory agent: incident + prior memory
4. planner: shared memory + required output fields + forbidden actions
5. reviewer: deterministic safety/completeness checks
6. repair: only if reviewer finds issues

The score is computed from the actual generated agent outputs and final plan.

In [ ]:
live_harness = run_live_hand_built_lane(scenario, model_name=HARNESS_MODEL)
print_result(console, live_harness)

print("\nACTUAL AGENT OUTPUTS")
print(live_harness.final_answer)

## 10. Inspect The Shared Memory

This is the harness value in concrete form.

The workflow did not just produce prose. It produced operational state that can be checked, audited, and reused.

In [ ]:
print("Incident facts")
pprint(live_harness.memory.incident_facts)

print("\nEvidence gathered from the log-investigator output")
for item in live_harness.memory.evidence:
    print("-", item)

print("\nRunbook constraints gathered from the runbook-agent output")
for item in live_harness.memory.runbook_steps:
    print("-", item)

print("\nPrior lessons gathered from the memory-agent output")
for item in live_harness.memory.prior_lessons:
    print("-", item)

print("\nReviewer objections")
print(live_harness.memory.reviewer_objections or "None")

print("\nFinal plan")
pprint(live_harness.memory.final_plan)

## 11. Live Side-By-Side Result

This is the only scorecard to show as evidence.

Both rows come from live Ollama Cloud calls. The difference is the harness, not a hardcoded result.

In [ ]:
print_comparison(console, [live_raw, live_harness])

## 12. Explain The Difference In Plain English

Use this in the meeting after the live scorecard:

- The weak lane asked a strong model to improvise from an incomplete view of the world.
- The good lane decomposed the work, gave each agent the right controlled context, accumulated shared memory, and checked the final result against policy.
- The scorecard is not grading writing style. It is grading production requirements: evidence, runbook use, safety, memory, completeness.

That is harness engineering.

# Spectrum Of Harness Engineering

Now explain the broader industry movement without showing fake results.

| Level | What we can show today | What it means |
| --- | --- | --- |
| Raw model | Live weak-harness lane above | Model quality alone is not operational reliability. |
| Hand-built harness | Live good-harness lane above | We can build controls explicitly. |
| SDK harness, e.g. Strands | Architecture mapping and next implementation step | Frameworks are packaging tools, hooks, memory, sessions, and multi-agent orchestration. |
| Provider harness | Adapter boundary and future integration | Provider-specific protocol quirks can become plug-and-play. |

Do not run static Strands or DeepSeek scores as proof. Present them as the spectrum roadmap unless they are wired to live calls.

# Optional Smoke Test

Only use this for your own pre-demo sanity check, not as a management proof.

```bash
harness-demo compare --scenario incident-response
python -m pytest -p no:cacheprovider
```

Those commands validate the repo wiring. The live evidence is above.

# Closing Narrative

The medium model is not magically smarter. It performs better because the harness gives it:

- controlled context
- tool boundaries
- shared memory
- policy/runbook grounding
- reviewer checks
- objective sensors
- repair loop
- repeatable scorecard

That is the difference between a chat answer and a production AI workflow.